In [2]:
# ============================================================
# NEO PROJECT: REPRODUCIBLE EVALUATION OF SIX MISSING-DATA METHODS
# ============================================================

# 1. IMPORT LIBRARIES
import os
import warnings
import zipfile
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# 2. CONFIGURATION
RANDOM_STATE = 42
TEST_SIZE = 0.20

FEATURES = ['a', 'e', 'i', 'om', 'w', 'diameter']
ORBITAL_FEATURES = ['a', 'e', 'i', 'om', 'w']

OUTPUT_DIR = '/content/neo_evaluation_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. LOAD DATASET DIRECTLY FROM GITHUB
DATA_URL = (
    "https://raw.githubusercontent.com/meherhasija/"
    "neo-missingness-lab/refs/heads/main/"
    "data/neo_asteroid_dataset.csv"
)

df = pd.read_csv(DATA_URL)

print("=" * 65)
print("DATASET INFORMATION")
print("=" * 65)
print("Dataset shape:", df.shape)
print("Missing diameter:", df['diameter'].isna().sum())
print("Missing PHA labels:", df['pha'].isna().sum())

# 4. DATA CLEANING
df = df.dropna(subset=['pha']).copy()

df['pha'] = df['pha'].astype(str).str.strip().str.upper()
df = df[df['pha'].isin(['Y', 'N'])].copy()

df['target'] = df['pha'].map({'N': 0, 'Y': 1})

X = df[FEATURES].copy()
y = df['target'].copy()

# 5. COMMON STRATIFIED TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTRAIN-TEST SPLIT")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Test PHA records:", int(y_test.sum()))

# Preserve original test indices
test_indices = X_test.index

# 6. HELPER FUNCTION: MODEL
def create_model():
    return HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        early_stopping=False
    )

# 7. HELPER FUNCTION: METRICS
def calculate_metrics(y_true, predictions, probabilities):
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(
            y_true, predictions, zero_division=0
        ),
        'Recall': recall_score(
            y_true, predictions, zero_division=0
        ),
        'F1': f1_score(
            y_true, predictions, zero_division=0
        ),
        'ROC_AUC': roc_auc_score(y_true, probabilities),
        'Average_Precision': average_precision_score(
            y_true, probabilities
        ),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }

# 8. STORAGE FOR RESULTS
all_predictions = []
all_metrics = []

# 9. METHOD 1: NATIVE MISSING VALUE HANDLING
print("\nRunning M1: Native Missing Value Handling")

model = create_model()
model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M1_Native',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 10. METHOD 2: MEDIAN IMPUTATION
print("Running M2: Median Imputation")

median_imputer = SimpleImputer(strategy='median')

X_train_median = pd.DataFrame(
    median_imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test_median = pd.DataFrame(
    median_imputer.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)

model = create_model()
model.fit(X_train_median, y_train)

prob = model.predict_proba(X_test_median)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M2_Median',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 11. METHOD 3: MEDIAN IMPUTATION + MISSINGNESS INDICATOR
print("Running M3: Median + Missingness Indicator")

X_train_m3 = X_train.copy()
X_test_m3 = X_test.copy()

X_train_m3['diameter_missing'] = (
    X_train['diameter'].isna().astype(int)
)
X_test_m3['diameter_missing'] = (
    X_test['diameter'].isna().astype(int)
)

m3_features = FEATURES + ['diameter_missing']

m3_imputer = SimpleImputer(strategy='median')

X_train_m3 = pd.DataFrame(
    m3_imputer.fit_transform(X_train_m3),
    columns=m3_features,
    index=X_train.index
)

X_test_m3 = pd.DataFrame(
    m3_imputer.transform(X_test_m3),
    columns=m3_features,
    index=X_test.index
)

model = create_model()
model.fit(X_train_m3, y_train)

prob = model.predict_proba(X_test_m3)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M3_Median_Indicator',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 12. METHOD 4: KNN IMPUTATION
print("Running M4: KNN Imputation")

# Fit orbital scaler on training data only
orbital_scaler = StandardScaler()

train_orbit_scaled = orbital_scaler.fit_transform(
    X_train[ORBITAL_FEATURES]
)

test_orbit_scaled = orbital_scaler.transform(
    X_test[ORBITAL_FEATURES]
)

# Training records with known diameter are donors
donor_mask = X_train['diameter'].notna()

donor_orbit = train_orbit_scaled[donor_mask.values]
donor_diameter = X_train.loc[
    donor_mask, 'diameter'
].values

if len(donor_diameter) < 5:
    raise ValueError("Not enough known training diameters for KNN.")

knn = NearestNeighbors(n_neighbors=5)
knn.fit(donor_orbit)

def knn_fill_diameter(data, scaled_orbit):
    result = data.copy()
    missing_mask = result['diameter'].isna()

    if missing_mask.any():
        distances, indices = knn.kneighbors(
            scaled_orbit[missing_mask.values]
        )

        neighbor_values = donor_diameter[indices]
        result.loc[missing_mask, 'diameter'] = (
            neighbor_values.mean(axis=1)
        )

    return result

X_train_knn = knn_fill_diameter(
    X_train, train_orbit_scaled
)

X_test_knn = knn_fill_diameter(
    X_test, test_orbit_scaled
)

model = create_model()
model.fit(X_train_knn, y_train)

prob = model.predict_proba(X_test_knn)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M4_KNN',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 13. METHOD 5: ITERATIVE IMPUTATION
print("Running M5: Iterative Imputation")

# Scaling and imputation are fitted using training data only
iterative_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=RANDOM_STATE,
        skip_complete=True
    ))
])

X_train_iterative = pd.DataFrame(
    iterative_pipeline.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test_iterative = pd.DataFrame(
    iterative_pipeline.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)

model = create_model()
model.fit(X_train_iterative, y_train)

prob = model.predict_proba(X_test_iterative)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M5_Iterative',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 14. METHOD 6: COMPLETE-CASE ANALYSIS
print("Running M6: Complete-Case Analysis")

train_complete_mask = X_train.notna().all(axis=1)
test_complete_mask = X_test.notna().all(axis=1)

X_train_complete = X_train.loc[train_complete_mask]
y_train_complete = y_train.loc[train_complete_mask]

X_test_complete = X_test.loc[test_complete_mask]
y_test_complete = y_test.loc[test_complete_mask]

model = create_model()
model.fit(X_train_complete, y_train_complete)

prob = model.predict_proba(X_test_complete)[:, 1]
pred = (prob >= 0.5).astype(int)

m6_predictions = pd.DataFrame({
    'method': 'M6_Complete_Case',
    'row_index': X_test_complete.index,
    'actual': y_test_complete.values,
    'predicted': pred,
    'probability': prob
})

all_predictions.append(m6_predictions)

print("M6 training records:", len(X_train_complete))
print("M6 testing records:", len(X_test_complete))

# 15. COMBINE PREDICTIONS
predictions_df = pd.concat(
    all_predictions, ignore_index=True
)

# 16. DEFINE TEST SUBGROUPS
measured_indices = X_test.index[
    X_test['diameter'].notna()
]

missing_indices = X_test.index[
    X_test['diameter'].isna()
]

subgroups = {
    'Full_Test_Set': list(X_test.index),
    'Measured_Diameter': list(measured_indices),
    'Missing_Diameter': list(missing_indices)
}

# 17. EVALUATE EVERY METHOD ON EACH VALID SUBGROUP
methods = predictions_df['method'].unique()


for method in methods:
    method_data = predictions_df[
        predictions_df['method'] == method
    ]

    for subgroup_name, subgroup_indices in subgroups.items():

        # M6 is a complete-case model.
        # It must not be reported as evaluating the full test set.
        if method == 'M6_Complete_Case':
            if subgroup_name == 'Full_Test_Set':
                continue

            if subgroup_name == 'Missing_Diameter':
                continue

        subset = method_data[
            method_data['row_index'].isin(subgroup_indices)
        ]

        if len(subset) == 0:
            continue

        y_true = subset['actual']
        y_pred = subset['predicted']
        y_prob = subset['probability']

        metrics = calculate_metrics(
            y_true, y_pred, y_prob
        )

        metrics['Method'] = method
        metrics['Evaluation_Subset'] = subgroup_name
        metrics['Records'] = len(subset)
        metrics['PHA_Prevalence'] = y_true.mean()

        all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics)

# Organize columns
column_order = [
    'Method', 'Evaluation_Subset', 'Records',
    'PHA_Prevalence', 'Accuracy', 'Precision',
    'Recall', 'F1', 'ROC_AUC', 'Average_Precision',
    'TN', 'FP', 'FN', 'TP'
]

metrics_df = metrics_df[column_order]

# 18. SAVE OUTPUT FILES
predictions_path = os.path.join(
    OUTPUT_DIR, 'all_methods_predictions.csv'
)

metrics_path = os.path.join(
    OUTPUT_DIR, 'all_methods_metrics.csv'
)

predictions_df.to_csv(predictions_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

# Separate subgroup metrics
for subgroup in metrics_df['Evaluation_Subset'].unique():
    subgroup_df = metrics_df[
        metrics_df['Evaluation_Subset'] == subgroup
    ]
    subgroup_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f'{subgroup.lower()}_metrics.csv'
        ),
        index=False
    )

# 19. SAVE DATASET AND SPLIT INFORMATION
split_info = pd.DataFrame({
    'row_index': test_indices,
    'actual': y_test.values,
    'diameter_available': X_test['diameter'].notna().values
})

split_info.to_csv(
    os.path.join(OUTPUT_DIR, 'test_split_information.csv'),
    index=False
)

# 20. DISPLAY RESULTS
print("\n" + "=" * 80)
print("FINAL RESULTS: ALL METHODS AND TEST SUBGROUPS")
print("=" * 80)

display(
    metrics_df.round(4).sort_values(
        ['Evaluation_Subset', 'Method']
    ).reset_index(drop=True)
)

print("\nTEST SUBSET COUNTS")
print("Full test set:", len(X_test))
print("Measured diameter:", len(measured_indices))
print("Missing diameter:", len(missing_indices))

print("\nMISSINGNESS DISTRIBUTION")
print("Training diameter missing:",
      X_train['diameter'].isna().sum())
print("Testing diameter missing:",
      X_test['diameter'].isna().sum())

# 21. CREATE ZIP OF ALL RESULTS
zip_path = '/content/neo_evaluation_results.zip'

with zipfile.ZipFile(zip_path, 'w',
                     zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(OUTPUT_DIR):
        filepath = os.path.join(OUTPUT_DIR, filename)
        zipf.write(filepath, arcname=filename)

print("\n" + "=" * 80)
print("EVALUATION COMPLETED")
print("=" * 80)
print("Results folder:", OUTPUT_DIR)
print("ZIP file:", zip_path)

print("\nDownload the ZIP file using:")
print("from google.colab import files")
print("files.download('/content/neo_evaluation_results.zip')")



DATASET INFORMATION
Dataset shape: (42815, 21)
Missing diameter: 41569
Missing PHA labels: 135

TRAIN-TEST SPLIT
Training records: 34144
Testing records: 8536
Test PHA records: 510

Running M1: Native Missing Value Handling
Running M2: Median Imputation
Running M3: Median + Missingness Indicator
Running M4: KNN Imputation
Running M5: Iterative Imputation
Running M6: Complete-Case Analysis
M6 training records: 999
M6 testing records: 247

FINAL RESULTS: ALL METHODS AND TEST SUBGROUPS


,Method,Evaluation_Subset,Records,PHA_Prevalence,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,TN,FP,FN,TP
0,M1_Native,Full_Test_Set,8536,0.0597,0.7733,0.1631,0.6765,0.2629,0.8135,0.2653,6256,1770,165,345
1,M2_Median,Full_Test_Set,8536,0.0597,0.7691,0.1604,0.6765,0.2593,0.8155,0.2628,6220,1806,165,345
2,M3_Median_Indicator,Full_Test_Set,8536,0.0597,0.7700,0.1584,0.6608,0.2556,0.8129,0.2781,6236,1790,173,337
3,M4_KNN,Full_Test_Set,8536,0.0597,0.7623,0.1511,0.6451,0.2449,0.7937,0.2028,6178,1848,181,329
4,M5_Iterative,Full_Test_Set,8536,0.0597,0.7649,0.1555,0.6627,0.2520,0.8030,0.2068,6191,1835,172,338
5,M1_Native,Measured_Diameter,247,0.2996,0.6113,0.4295,0.9054,0.5826,0.8002,0.6487,84,89,7,67
6,M2_Median,Measured_Diameter,247,0.2996,0.5870,0.4125,0.8919,0.5641,0.7820,0.6086,79,94,8,66
7,M3_Median_Indicator,Measured_Diameter,247,0.2996,0.6073,0.4286,0.9324,0.5872,0.8103,0.6695,81,92,5,69
8,M4_KNN,Measured_Diameter,247,0.2996,0.6235,0.4128,0.6081,0.4918,0.7081,0.4540,109,64,29,45
9,M5_Iterative,Measured_Diameter,247,0.2996,0.6316,0.4341,0.7568,0.5517,0.7264,0.4685,100,73,18,56



TEST SUBSET COUNTS
Full test set: 8536
Measured diameter: 247
Missing diameter: 8289

MISSINGNESS DISTRIBUTION
Training diameter missing: 33145
Testing diameter missing: 8289

EVALUATION COMPLETED
Results folder: /content/neo_evaluation_results
ZIP file: /content/neo_evaluation_results.zip

Download the ZIP file using:
from google.colab import files
files.download('/content/neo_evaluation_results.zip')


In [3]:
from google.colab import files

files.download('/content/neo_evaluation_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>